# Ridge and Lasso: Charging for Big Coefficients

A flexible model can bend to fit the training data closely, and that's exactly how it overfits. This notebook starts with a way to make a linear model flexible (polynomial regression), shows what goes wrong, and then fixes it with **ridge regression** and its cousin the **lasso** (ISL §6.2): keep the flexibility, but charge the model for using big coefficients. As with $K$ in KNN, the strength of the charge is chosen by cross-validation.

**Where this sits in modern AI.** Ridge's penalty is one of the most common forms of regularization in neural networks, where it is called **weight decay**. Large language models are trained with it (their optimizers apply a close variant), and by the end of this notebook you'll know what it does.

### By the end, you'll be able to

1. Fit a polynomial by extending linear regression with features made from $x$, and say what happens to the fit as the degree grows.
2. Recognize overfitting in two ways: a training error far below the test error, and fits that change wildly from one sample to the next.
3. Write the RSS, then the ridge loss $L(\beta)$, and read each in words.
4. Predict how ridge's coefficients and fit change as $\lambda$ grows, and describe $\lambda$ as a flexibility setting in bias–variance terms.
5. Explain why features must be standardized before ridge (the same applies to lasso), and why the intercept isn't penalized.
6. Explain a second benefit of ridge: it steadies the coefficients when two features are nearly copies of each other, and what it gives up to do so.
7. Read the two-coefficient picture of ridge: the loss surface, and how its bottom moves as $\lambda$ grows.
8. Say how the lasso differs from ridge: it sets some coefficients **exactly** to zero, so it also selects features.
9. Describe ridge and the lasso as fitting within a budget (a circle for ridge, a diamond for the lasso), and explain why the diamond's corners set coefficients exactly to zero.
10. Choose $\lambda$ by cross-validation, then refit.
11. Put ridge, lasso, and tree pruning side by side as one shape, and translate between ISL's symbols and scikit-learn's names ($\alpha$, $\lambda$, `ccp_alpha`, `alpha`, `C`).

> **How to read the code in this notebook.** There are two kinds of code cells.
>
> - **Code worth reading.** Cells that use standard libraries, such as numpy (`np.`), pandas (`pd.`), and scikit-learn, do everyday tasks you'll do yourself. Each is introduced with **Code worth reading** in bold. Read these line by line.
> - **Demo cells.** Cells that import from `ch_6_and_10_demos_helpers` draw a plot or build an interactive demo. That file was written to keep the notebook on ideas, so you never need to open it.

In [ ]:
import numpy as np
import pandas as pd

<a id="section-1"></a>

## 1 · Fitting a curve: polynomial regression

Linear regression fits a line, $\hat y = \beta_0 + \beta_1 x$. To fit a curve, give the model more features made from $x$. A **polynomial of degree $d$** uses $x, x^2, \dots, x^d$:

$$
\hat y = \beta_0 + \beta_1 x + \beta_2 x^2 + \dots + \beta_d x^d
$$

This is still linear regression: the model is a linear combination of features, and the features happen to be powers of one variable. Nothing about the fitting changes. A higher degree gives a more flexible curve.

**Code worth reading.** Here are 20 points from a smooth curve plus noise. `PolynomialFeatures` builds the new columns. For three of the points (the first, middle, and last) with $d = 3$, the columns are $x$, $x^2$, and $x^3$:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

from ch_6_and_10_demos_helpers import wiggle_data

data = wiggle_data()                                 # 20 training points, 2,000 new points
x_column = data.x.reshape(-1, 1)                     # scikit-learn wants one column per feature

poly = PolynomialFeatures(degree=3, include_bias=False)
x_poly = poly.fit_transform(x_column)                # columns: x, x^2, x^3
pd.DataFrame(x_poly[[0, 10, 19]], columns=['x', 'x^2', 'x^3'],
             index=['first point', 'middle point', 'last point']).round(3)

**Code worth reading.** Fitting is the same `LinearRegression` as in Chapter 3, on the new columns. It finds an intercept and one coefficient per column:

<details>
<summary>Why is it still called <i>linear</i> regression if it fits a cubic?</summary>

"Linear" describes the **coefficients**, not $x$. The prediction $\beta_0 + \beta_1 x + \beta_2 x^2 + \beta_3 x^3$ multiplies each column by its coefficient and adds the results, so it is a linear combination of the columns. The columns can be anything computed from the data beforehand ($x^2$, $x^3$, $\log x$, ...). Least squares never looks inside them; it only sees a table of numbers, and solves for the $\beta$'s exactly as in Chapter 3, with the same formula $(X^\top X)^{-1} X^\top y$.

The fit is a curve when plotted against $x$, but a flat plane in the space whose axes are the columns $x$, $x^2$, $x^3$. A model like $\hat y = \beta_0 + x^{\beta_1}$, where a coefficient sits inside the power, would **not** be linear regression.

</details>

In [ ]:
cubic = LinearRegression().fit(x_poly, data.y)
print(f'y-hat = {cubic.intercept_:.2f} + ({cubic.coef_[0]:.2f})x + ({cubic.coef_[1]:.2f})x^2 '
      f'+ ({cubic.coef_[2]:.2f})x^3')

Here are the fits for degree 1 (a line), degree 3, and degree 12, on the same 20 points.

In [ ]:
from ch_6_and_10_demos_helpers import plot_polynomial_fits

plot_polynomial_fits(data, degrees=(1, 3, 12));

<a id="section-2"></a>

## 2 · Overfitting, again

The degree-1 line is too rigid (high bias), and the degree-3 curve follows the truth well. The degree-12 curve passes close to nearly every training point, but between and beyond them it swings wildly. This is overfitting, and it shows up in two ways.

**1. Training error is far below test error.** **Code worth reading.** The training MSE is measured on the 20 points; the next column is measured on 2,000 new points from the same curve. (The noise we added has variance 0.09, so no model can do much better than that.) The last column is the size of the largest coefficient, which matters in the next section.

**Predict first:** which degree will have the lowest training MSE? The lowest MSE on new points?

In [ ]:
from ch_6_and_10_demos_helpers import polynomial_error_table

pd.DataFrame(polynomial_error_table(data, degrees=[1, 3, 5, 12])).T.round(3)

Training error only falls as the degree grows. Test error falls, and then explodes: the degree-12 fit has the lowest training error and the worst test error by far.

**2. The fit changes wildly from one sample to the next.** Draw eight fresh samples of 20 points, fit each, and overlay the curves. A model that has learned the pattern gives nearly the same curve every time. A model that has learned the noise doesn't.

**Predict first:** where will the eight degree-12 curves disagree the most, and why there?

In [ ]:
from ch_6_and_10_demos_helpers import plot_resampled_fits

plot_resampled_fits(degrees=(3, 12));

<details>
<summary>Answer</summary>

**At the two ends.** In the middle, training points on both sides pin the curve down. Near the edges, there are training points on only one side, so a flexible curve is free to whip up or down, and where it ends up depends on the noise in the few points nearby. (In the middle the eight curves differ by about 0.2 in height; at the ends, by several units or more.) This is high **variance**.

</details>

<a id="section-3"></a>

## 3 · Charging for big coefficients: ridge

**How can we get the benefits of a flexible model but avoid the overfitting?** We don't want to give up the degree-12 model's ability to bend where the data really bends. We want to stop it from bending *wildly* and "memorizing" noise in the data.

Look at how the wild curve is built. It's a sum of terms like $\beta_{12} x^{12}$ and $\beta_{11} x^{11}$, and to make the curve swing back and forth between points, least squares picks huge coefficients of opposite signs that nearly cancel in the middle and don't cancel at the ends. Here is the last column of the error table from [Section 2](#section-2):

| Degree | 1 | 3 | 5 | 12 |
| --- | --- | --- | --- | --- |
| Largest coefficient | 1.6 | 31 | 143 | 31,602 |

So a wild curve needs **big coefficients**. A fix follows: **make big coefficients cost something.**

To see how, start with what least squares minimizes, the **residual sum of squares**:

$$
\text{RSS} = \sum_{i=1}^{n} \big( y_i - \hat y_i \big)^2
$$

- For each training point $i$, compute the gap between the true $y_i$ and the prediction $\hat y_i$, square it, and add them up. 
- Least squares chooses the coefficients that make this as small as possible. 

With $p$ features we bring in their coefficients: $\hat y_i = \beta_0 + \sum_{j=1}^{p} \beta_j x_{ij}$. 

Substituting gives the longer form you saw in Chapter 3:

$$
\text{RSS} = \sum_{i=1}^{n} \Big( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \Big)^2
$$

It's the same formula; only the prediction has been written out. (For our polynomial, $p = 12$, and the features are powers of $x$: $x_{i1} = x_i$, $x_{i2} = x_i^2$, …, $x_{i,12} = x_i^{12}$.)

**Ask:** How might we modify the RSS to produce a loss that mitigates/reduces overfitting?

<details>
<summary>Hint</summary>

- Right now, what is the only thing least squares rewards? Does anything in the RSS stop the coefficients from getting huge?
- Look back at the table. What do the wild curves have that the calm ones don't?
- How could you measure "how big the coefficients are"? Can you represent it as a single number you could add to RSS? 
- If you added that number to the RSS, what would minimizing the sum have to trade off (or balance)? Could you introduce a "knob" to control how much each part (RSS vs your measurement) contributes to the total loss?

</details>

<details>
<summary>Answer</summary>

**Add a second term that charges for big coefficients**, with a "dial" (coefficient) for how much it impacts the total:

$$
\text{loss} = \text{RSS} + \text{strength} \times \text{(size of the coefficients)}
$$

Minimizing the RSS alone rewards fitting the training points and nothing else. With the added charge, the fit has to trade closeness to the training points against the size of its coefficients, so it can't use huge coefficients to chase the noise. The rest of this notebook explores two common ways to measure the size: **ridge regression** adds up the squared coefficients, and **the lasso** adds up their absolute values.

</details>

**Ridge regression** (ISL equation 6.5) adds a charge for big coefficients:

$$
L(\beta) = \underbrace{\sum_{i=1}^{n} \Big( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \Big)^2}_{\text{RSS: how badly the fit misses the training points}} \;+\; \underbrace{\lambda \sum_{j=1}^{p} \beta_j^2}_{\text{penalty: how big the coefficients are}}
$$

As in the gradient-descent notebook, we call this the **loss**, $L(\beta)$: one number that measures how bad a choice of coefficients is. ($\beta$ stands for all the coefficients together.) Ridge chooses the coefficients that make the loss as small as possible, and its two parts pull in opposite directions:

- The **RSS** is small when the curve passes close to the training points. For the degree-12 curve on our 20 points, making the RSS as small as possible required the huge coefficients in the table above.
- The **penalty** is $\lambda$ times a sum of squared coefficients, so it is small only when every coefficient is small. (The intercept $\beta_0$ isn't in it.)

So ridge can't make both parts as small as possible at once. It settles for a compromise: a curve that misses the points by a bit more, in exchange for much smaller coefficients, and smaller coefficients mean a calmer curve. $\lambda \ge 0$ is the **strength** of the penalty, and we choose it. It sets the exchange rate: with a large $\lambda$, coefficient size costs a lot, so ridge accepts a bigger RSS to keep the coefficients small. The penalty term is called a **shrinkage penalty**, because it shrinks the coefficients toward zero, and ridge and the lasso are called **shrinkage methods**. (scikit-learn calls $\lambda$ `alpha`.)

**Ask:** At $\lambda = 0$, what model does ridge give? At a huge $\lambda$?

<details>
<summary>Answer</summary>

At $\lambda = 0$ there's no penalty, so ridge gives ordinary least squares. At a huge $\lambda$, any nonzero coefficient costs more than it could save in RSS, so all the coefficients go to (nearly) zero, and the model predicts about the average response for everyone. In between is a dial from the wild curve to a flat line.

</details>

<a id="demo-f"></a>

## 4 · Turning the dial

### Demo F: $\lambda$ as a flexibility dial

The same 20 points and the same 12 features from Section 1, now fit with ridge. The slider is $\lambda$. The right panel shows the 12 coefficients, on a scale that shows both huge and small values. The slider starts at $\lambda = 0$, plain least squares, which is the degree-12 curve from Section 1. (The demo standardizes the 12 features first; Section 5 says why. On standardized features the least squares coefficients come out in the thousands rather than near 30,000, but they are just as wild.)

In [ ]:
from ch_6_and_10_demos_helpers import ridge_wiggle_demo

ridge_wiggle_demo(size_percent=80)

**Ask:** As I slide $\lambda$ up from the left, what happens to the curve, and to the bars? What happens to the training MSE and to the MSE on new points?

<details>
<summary>Answer</summary>

The bars shrink toward zero, and the curve calms down: from the wild degree-12 curve that chases the points (and shoots off at the edges), to one close to the true curve, to a nearly flat line at very large $\lambda$. **The training MSE only rises.** The error on new points falls, then rises: a small $\lambda$ is the flexible end, a large $\lambda$ the rigid end, and somewhere between is a good balance. That's the benefit of a flexible model without the overfitting.

</details>

<a id="section-5"></a>

## 5 · Real Data: Standardize first

The *penalty* charges each coefficient proportional to its magnitude. But a coefficient's size depends on its feature's units. Measure a player's career at-bats in thousands instead of single at-bats, and its coefficient grows 1,000 times, so the penalty would charge it 1,000,000 times more. **Standardizing** puts every feature in the same unit, standard deviations, so the regularization penalty treats them consistently. This is the same standardization technique from the gradient-descent notebook, (but for a different reason) where standardizing `TV` made the loss surface ("bowl") round, and in Chapter 4's optional lab.

Trees don't need standardizing, even with regularization. A split only asks "which side of $s$ is this point on?", and changing a feature's units moves $s$ along with the points.

**Code worth reading.** From here we use all of Hitters: 19 features to predict salary (in thousands of dollars), dropping the players with no salary, as in *Trees and Regularization*. Three columns are categories with two values each (`League`, `Division`, `NewLeague`).

- `make_column_transformer` applies different steps to different columns: here, `StandardScaler` to the numeric columns and `OneHotEncoder` to the categorical ones.
- `OneHotEncoder` turns a category into 0/1 columns. With `drop='if_binary'`, a two-valued column like `League` (A or N) becomes a single column, `League_N`, that is 1 for league N and 0 for league A.
- `make_pipeline` chains the preprocessing and the model, so `fit` runs every step in order, and cross-validation later repeats every step inside each fold.

In [ ]:
from ISLP import load_data
from sklearn.compose import make_column_transformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

hitters = load_data('Hitters').dropna(subset=['Salary']) #model can't learn from rows with a missing target value
X = hitters.drop(columns='Salary')
y = hitters['Salary']                                # in thousands of dollars

categorical_columns = ['League', 'Division', 'NewLeague']
numeric_columns = [col for col in X.columns if col not in categorical_columns]

preprocess = make_column_transformer(
    (StandardScaler(), numeric_columns),
    (OneHotEncoder(drop='if_binary'), categorical_columns),
    verbose_feature_names_out=False)                 # keep the original column names
ridge = make_pipeline(preprocess, Ridge(alpha=10))
ridge.fit(X, y)

ridge_coefficients = ridge[-1].coef_                # ridge[-1] is the pipeline's last step, the Ridge model
pd.Series(ridge_coefficients, index=preprocess.get_feature_names_out()).round(1)

Each coefficient is the change in predicted salary (in thousands) for a one-standard-deviation increase in that feature, with the others held fixed.

<a id="section-6"></a>

## 6 · Another benefit: steadier coefficients when features are near-copies

Like least squares, ridge doesn't need a step-by-step search: a formula gives its coefficients directly. (If you've seen the least squares formula $\hat\beta = (X^\top X)^{-1} X^\top y$, ridge's is the same with $\lambda$ added in one place: $(X^\top X + \lambda I)^{-1} X^\top y$. You won't need to work with either one.)

Reducing overfitting is the main reason to use ridge. A second benefit shows up when features are nearly copies of each other, like a player's `Hits` and `CHits` if everyone had played for one season. Least squares can't decide how to split the credit between near-copies, so its coefficients swing from sample to sample. Here are six samples of 30 rows. In every sample, the truth is $y = x_1 + x_2 + \text{noise}$, so the true coefficients are 1 and 1.

**Code worth reading.** Read the loop at the bottom: it draws six samples and fits each one twice, with $\lambda = 0$ (least squares) and $\lambda = 5$ (ridge). `fit_ridge` computes ridge's formula; you don't need to follow its details.

In [ ]:
rng = np.random.default_rng(1)

def make_sample(n=30):
    x1 = rng.normal(size=n)
    x2 = x1 + rng.normal(scale=0.05, size=n)         # x2 is nearly a copy of x1
    y = x1 + x2 + rng.normal(size=n)                 # true coefficients: 1 and 1
    return np.column_stack([x1, x2]), y

def fit_ridge(X, y, lam):                            # ridge's formula; lam = 0 is least squares
    Xc, yc = X - X.mean(axis=0), y - y.mean()        # centering handles the intercept
    return np.linalg.solve(Xc.T @ Xc + lam * np.eye(2), Xc.T @ yc)

samples = [make_sample() for _ in range(6)]
rows = []
for k, (X_k, y_k) in enumerate(samples, start=1):
    b_ols, b_ridge = fit_ridge(X_k, y_k, 0), fit_ridge(X_k, y_k, 5)
    rows.append({'sample': k,
                 'least squares b1': b_ols[0], 'least squares b2': b_ols[1],
                 'least squares b1+b2': b_ols.sum(),
                 'ridge b1': b_ridge[0], 'ridge b2': b_ridge[1]})
pd.DataFrame(rows).set_index('sample').round(2)

Least squares swings wildly from sample to sample: one sample gives roughly $+7.6$ and $-5.9$, another $-3.4$ and $+5.4$. Yet the **sum** $b_1 + b_2$ stays near 2 every time. The data can tell how much the two features matter *together*, but not how to split the credit between them.

Ridge splits it about evenly, every time. Here that's close to the truth, but ridge would split it evenly even if the truth were 2 and 0, because the data can't tell the difference. What ridge gives you is **stability**, not the true split.

Notice that least squares' *predictions* are fine here: since $x_2 \approx x_1$, it predicts about $(b_1 + b_2)\,x_1$, and that sum is steady. The swinging coefficients hurt when you try to read the model ("which feature matters?"), and when a new point has $x_1$ and $x_2$ that disagree.

This is also part of what went wrong with the degree-12 curve. Its features are near-copies too ($x^{11}$ and $x^{12}$ have a correlation of 0.999 on our data), which is how least squares ended up with huge coefficients of opposite signs.

**Ask:** In this example, ridge's coefficients are a bit smaller than the truth (about 0.9 instead of 1). Is that a problem?

<details>
<summary>Answer</summary>

It's the price of the trade. Ridge is biased: it pulls the coefficients slightly toward zero. In exchange, its coefficients barely move from sample to sample, while least squares' swing by several units. This is **bias–variance again, with $\lambda$ as the dial**: in Section 4 it calmed the curve, and here it steadies the coefficients.

</details>

<a id="section-7"></a>

## 7 · A picture of ridge

Ridge makes the loss, $L(\beta) = \text{RSS} + \lambda \times \text{penalty}$, as small as possible. This section draws the loss so you can see where ridge's answer comes from. Two slider demos build the picture: the loss with one coefficient, then with two.

The data: 40 rows and two standardized features with a correlation of about 0.9, a milder version of Section 6's near-copies. The truth is $y = 2x_1 + 0.3x_2 + \text{noise}$. To keep the number of coefficients small enough to draw, $y$ is centered, so there's no intercept.

**Code worth reading.** `fit_ridge` is the function from Section 6: least squares first, then ridge at $\lambda = 30$.

In [ ]:
from ch_6_and_10_demos_helpers import two_feature_data

X_pic, y_pic = two_feature_data()
print('least squares:     ', fit_ridge(X_pic, y_pic, 0).round(2))
print('ridge, lambda = 30:', fit_ridge(X_pic, y_pic, 30).round(2))

<a id="demo-k"></a>

### Demo K: One coefficient, seen from the side

Start with a single coefficient: predict $y$ from $x_1$ alone, so the model is $\hat y = \beta x_1$. Each cost is then a curve over $\beta$, and its height is the cost:

- **Gray: the RSS.** It comes from the data. Its lowest point is the least squares coefficient, 2.26.
- **Orange: the penalty, $\lambda \beta^2$.** It depends only on $\beta$, not on the data. Its lowest point is 0. $\lambda$ sets how steep it is.
- **Blue: the loss, their sum.** Ridge's answer, the star, is its lowest point.

The dotted line marks the star's $\beta$. The gray and orange dots are the RSS and the penalty at that $\beta$; their heights add up to the star's height. The readout gives the numbers.

**Code worth reading.** `X_pic[:, 0]` is the first column, $x_1$.

**Predict first:** Drag $\lambda$ from 0 to the right. What happens to the orange curve? Does the star go all the way to 0?

In [ ]:
from ch_6_and_10_demos_helpers import penalty_side_view_demo

penalty_side_view_demo(X_pic[:, 0], y_pic, size_percent=80)

<details>
<summary>What to notice (Read this!)</summary>

- **Convex Curves:** The RSS (gray) and the penalty (orange) are both **bowl-shaped**. At small $\lambda$ this is hard to see. Make it bigger and the shape is obvious.
- **At $\lambda = 0$,** the orange curve lies flat along the bottom (the penalty is 0 everywhere). The loss is just the RSS, and the star sits on least squares.
- **As $\lambda$ grows,** the orange curve gets steeper, and the star slides toward 0.
- **The star stops between least squares and 0.** Sliding it closer to 0 would lower the penalty, but the RSS climbs faster there. At the star, any step in either direction raises the loss. That balance is what ridge's answer is.
- **The star never reaches 0.** Even at $\lambda = 1{,}585$, the coefficient is about 0.06.

</details>

**Ask:** Watch the readout in Demo K as you drag $\lambda$. The loss is smallest at $\lambda = 0$ (8.1), and it grows as $\lambda$ grows (89.7 at $\lambda = 27$). So what's the point of the penalty? Should we just set $\lambda = 0$ to get the smallest loss?

<details>
<summary>Hint</summary>

Which data are the RSS and the loss computed from: the training data or new data? Which of the two do we actually care about?

</details>

<details>
<summary>Answer</summary>

No, we should *not* just set $\lambda = 0$. Two reasons:

1. **The loss is computed on the training data.** The smallest possible training loss is exactly what an overfit model achieves; the degree-12 curve in Section 2 had the lowest training error and by far the worst error on new points. We add the penalty to do better on **new** data, and that can cost a little on the training data.
2. **Losses at different $\lambda$ values aren't comparable.** Each $\lambda$ gives a different loss: a bigger $\lambda$ adds a bigger charge for the same coefficients, so its lowest value can only go up. Fix one $\lambda$ value and the loss determines (enforces) which coefficients to pick. It can't tell you which $\lambda$ to pick.

To choose $\lambda$, measure error on data the model wasn't fit to, with no penalty added: that's cross-validation, in [Section 9](#section-9).

</details>

<a id="demo-l"></a>

### Demo L: The loss surface with two coefficients

Back to both features. Now every pair $(\beta_1, \beta_2)$ is a point on a plane, and the loss $L(\beta) = \text{RSS} + \lambda (\beta_1^2 + \beta_2^2)$ gives each point a height: its RSS plus its penalty, added together. So the loss is a **surface** over the plane, like the error bowls in the gradient-descent notebook.

- **Left: the surface in 3D.** It's cut off at a height of 450 so its bottom stays in view.
- **Right: the same surface seen from above.** Color shows height (darker is lower). Each black curve is a **contour**: it connects points with the same loss, like the lines on a topographic map, and its number is that loss value.

The star is the bottom of the surface, ridge's answer. The colors use the same scale at every $\lambda$, and the slider is $\lambda$.

**Predict first:** As $\lambda$ grows, where does the bottom of the surface move? Does it get higher or lower? Does the surface's shape change?

In [ ]:
from ch_6_and_10_demos_helpers import loss_surface_demo

loss_surface_demo(X_pic, y_pic, size_percent=80)

<details>
<summary>What to notice</summary>

- **At $\lambda = 0$,** the loss is just the RSS: a long, narrow valley whose bottom is the least squares point, with a loss of 7.2.
- **As $\lambda$ grows, the bottom moves toward $(0, 0)$,** and the valley turns into a round, steep bowl. The penalty's round shape takes over from the RSS's long one.
- **The bottom rises:** the lowest loss is 7.2 at $\lambda = 0$, 42.5 at $\lambda = 14$, and 193.7 at $\lambda = 717$. This is the Ask after Demo K, in two dimensions: a bigger $\lambda$ charges more for the same coefficients, so the whole surface sits higher.
- **The surface is one bowl, not two.** Every height on it is an RSS and a penalty added together.

</details>

<a id="section-8"></a>

## 8 · The lasso: a different charge

**The lasso** (ISL equation 6.7) is ridge with one change: it charges the sum of the coefficients' *absolute values* instead of their squares:

$$
L(\beta) = \sum_{i=1}^{n} \Big( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \Big)^2 \;+\; \lambda \sum_{j=1}^{p} \lvert \beta_j \rvert
$$

It looks like a small change, but it behaves differently. (Ridge's coefficients come from a formula, as Section 6 mentions; the lasso's have no formula and are found step by step.)

<a id="demo-g"></a>

### Demo G: Coefficient paths

Back to Hitters, with all 19 standardized features (ISL Figures 6.4 and 6.6). Each line's coefficient value, plotted against $\lambda$. Toggle between **Ridge** and **Lasso**, and watch the readout.

**Code worth reading.** `preprocess.fit_transform(X)` standardizes and encodes the features once, so the demo can fit many models quickly.

In [ ]:
X_std = preprocess.fit_transform(X)                  # standardized, one-hot encoded
feature_names = list(preprocess.get_feature_names_out())

In [ ]:
from ch_6_and_10_demos_helpers import coefficient_path_demo

coefficient_path_demo(X_std, y, feature_names, size_percent=80)

Ridge shrinks every coefficient toward zero, but none of them reach it: at every $\lambda$, all 19 features stay in the model. The lasso sets coefficients **exactly** to zero, one after another, as $\lambda$ grows. A feature with a zero coefficient is out of the model, so the lasso does **feature selection** as part of fitting. That's why it's called an *embedded* feature-selection method.

One more scikit-learn detail: `Lasso` divides the RSS by $2n$ before adding the penalty, and `Ridge` doesn't. So the same `alpha` number means a much stronger penalty in `Lasso` than in `Ridge`. Never copy a value from one to the other; choose each by cross-validation (Section 9).

**Code worth reading.** Fit the lasso at one $\lambda$ and list the features it keeps.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=10).fit(X_std, y)
kept = pd.Series(lasso.coef_, index=feature_names)
print(f'Kept {np.sum(kept != 0)} of {len(kept)} features:')
kept[kept != 0].round(1)

**Ask:** You suspect only a handful of these 19 features matter. Which penalty would you try first, and why?

<details>
<summary>Answer</summary>

The lasso. It can drop the features that don't matter (set their coefficients to exactly 0), leaving a smaller model that's easier to read. Ridge would keep all 19. (If many features each matter a little, ridge often predicts better; cross-validation can settle it.)

</details>

**Ask:** Ridge and the lasso differ in one small place: squares versus absolute values. Yet in Demo G, the lasso's coefficients reach **exactly** zero, one after another, while ridge's only get close. Why would that one change make the difference? Make a guess before Demo M.

<a id="demo-m"></a>

### Demo M: Ridge and the lasso as a budget

Here's a second way to state both methods (ISL equations 6.8 and 6.9, and Figure 6.7). Think of the size of the coefficients as something you have a **budget** for:

- **Ridge:** choose a budget $s$. Consider only the models with $\beta_1^2 + \beta_2^2 \le s$, and among those, pick the one with the lowest RSS.
- **The lasso:** the same, but the budget limits $\lvert \beta_1 \rvert + \lvert \beta_2 \rvert \le s$.

Back to Section 7's two features. Every point in the plane is a model: a pair $(\beta_1, \beta_2)$. The shaded region holds **every model the budget allows**: a disk for ridge, a diamond for the lasso. Both are centered at $(0, 0)$, the model with both coefficients 0, which costs nothing. A bigger budget allows bigger coefficients, so it plays the role of a *smaller* $\lambda$. (The collapsible after this demo explains the match.)

The slider sets where both regions cross the axes, so the circle and the diamond are always the same size and only their shapes differ. The same size takes a different budget for each (ridge's $s$ is the square of where its circle crosses), so the readout gives each one's $s$.

Turn the layers on in this order:

1. **Regions only** (the starting setting). Drag the slider. Every point inside a region is a model that fits the budget.
2. **Add "RSS ovals."**
   - The ovals come from the data and are centered on least squares. All the models on one oval have the same RSS, and the RSS grows from each oval to the next, moving out from least squares. (They're the contours of Demo L at $\lambda = 0$, when the loss is just the RSS.)
   - They're long and thin because the features are correlated: moving along the long direction raises $\beta_1$ and lowers $\beta_2$ (or the reverse), and that barely changes the RSS. The two features rise and fall together, so shifting weight from one to the other leaves the predictions almost the same.
3. **Add "best model in the budget."** The star is the lowest-RSS model inside the region. Picture an oval growing out from least squares until it first reaches the region: the star is where they meet, and the dark oval is that oval. Shrink the budget and watch the lasso's panel.
4. **Add "path as the budget grows" and "closest point on the circle to least squares."**

**Predict first:** With a small budget, where on the diamond do you expect the lasso's best model to sit? Where on the circle do you expect ridge's?

In [ ]:
from ch_6_and_10_demos_helpers import budget_regions_demo

budget_regions_demo(X_pic, y_pic, size_percent=80)

<details>
<summary>What to notice</summary>

- **A big enough budget gives least squares.** Once a region reaches least squares (past about 2.0 on the slider for ridge, 2.3 for the lasso), the best model is least squares itself: no shrinking, the same as $\lambda = 0$.
- **Otherwise, the best model is on the region's edge, never inside.** From any model inside the region, a small step toward least squares lowers the RSS and still fits the budget. So the best model always spends the whole budget.
- **The oval through the star just touches the edge; it never crosses it.** If it crossed into the region, a slightly smaller oval, with a lower RSS, would still reach the region, and the star wouldn't be the best.
- **The lasso's answer lands on a corner.** The diamond's corners sit on the axes and stick out toward the ovals, so a growing oval often reaches a corner first. At a corner, one coefficient is **exactly** 0. Here, at 1.6 or below on the slider, the lasso's best model sits on the $\beta_1$ axis: $\beta_2 = 0$, so $x_2$ is out of the model. That answers the Ask.
- **A circle has no corners,** so an oval almost never first reaches it exactly on an axis. Ridge's coefficients get small but stay nonzero, as in Demo G.
- **Ridge's path curves.** As the budget shrinks from least squares, the star first moves toward $\beta_1 = \beta_2$: ridge evens out the credit between the correlated features, a change that barely raises the RSS. $\beta_2$ even grows at first, from 0.32 to about 0.76 (near 1.4 on the slider), before both coefficients shrink together. This is Section 6's near-copies story, drawn.
- **The hollow point isn't the star.** The best model is the lowest-RSS point on the circle, and that depends on the ovals' shape, not only on distance from least squares.
- **With many features,** the picture is the same in more dimensions. The lasso's region has corners and edges wherever some coefficients are 0, so it tends to zero out several at once. That's what Demo G's lasso paths show.

</details>

<details>
<summary>Budget or $\lambda$? Two ways to state the same method</summary>

Section 7 stated ridge with $\lambda$: make $\text{RSS} + \lambda \times \text{penalty}$ as small as possible. The budget states it with $s$: make the RSS as small as possible, with $\text{penalty} \le s$. The two give the same models. Each budget matches one $\lambda$, and the readout shows it. A bigger budget matches a smaller $\lambda$, and a budget big enough to include least squares matches $\lambda = 0$.

Why the same? Take ridge's answer at some $\lambda$, and draw the circle through it. Every point inside that circle has a penalty no bigger than the answer's. If one of them also had a lower RSS, it would have a lower loss, $\text{RSS} + \lambda \times \text{penalty}$, but the answer has the lowest loss. So the answer is the lowest-RSS point within that circle: the budget answer for that circle's $s$. The same argument works for the lasso's diamond.

So choosing $s$ chooses $\lambda$, and the reverse. scikit-learn, and the cross-validation in Section 9, use $\lambda$; the budget is the easier picture.

(The readout's $\lambda$ uses this notebook's formula, $\text{RSS} + \lambda \times \text{penalty}$. As Demo G notes, scikit-learn's `Lasso` scales its `alpha` differently.)

</details>

<a id="section-9"></a>
<a id="demo-i"></a>

## 9 · Choosing $\lambda$ by cross-validation

### Demo I: The cross-validation curve

How strong should the penalty be? The same question as $K$ in KNN, and the same answer: cross-validation. Ridge first, then the lasso.

**Code worth reading.** The same 10-fold `cross_val_score` loop as Lab 4, over a grid of $\lambda$ values. `np.logspace(-2, 5, 29)` makes 29 values from $10^{-2}$ to $10^{5}$, evenly spaced on a log scale. The whole pipeline goes into `cross_val_score`, so the standardizing is redone inside every fold, using only that fold's training players. `scoring='neg_mean_squared_error'` returns the MSE with a minus sign, so we flip it back, as in Lab 4.

<details>
<summary>Why a log scale?</summary>

We don't know in advance whether a good $\lambda$ is 0.1 or 1,000, so the grid has to cover many powers of ten. What changes the fit is multiplying $\lambda$ by some factor, not adding to it: going from 0.01 to 0.1 changes the fit about as much as going from 1,000 to 10,000.

An evenly spaced grid wastes its values. Twenty-nine values from 0.01 to 100,000 spaced evenly would be about 3,600 apart, so the second value would already be about 3,600, and everything below that would be skipped. A log-spaced grid instead multiplies by the same factor at each step (here about 1.78, so four steps per power of ten): 0.01, 0.018, 0.032, 0.056, 0.1, … , 100,000. Every power of ten gets the same number of values.

The plots below use a log scale on the $\lambda$ axis for the same reason.

</details>

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

folds = KFold(n_splits=10, shuffle=True, random_state=0)
ridge_lambdas = np.logspace(-2, 5, 29)

ridge_mse = []
for lam in ridge_lambdas:
    model = make_pipeline(preprocess, Ridge(alpha=lam))
    scores = cross_val_score(model, X, y, cv=folds, scoring='neg_mean_squared_error')
    ridge_mse.append(-scores.mean())

**Code worth reading.** Two reference points. `LinearRegression` is least squares, no penalty ($\lambda = 0$). `DummyRegressor` ignores the features and predicts the training players' average salary for everyone, which is where a huge $\lambda$ ends up.

In [ ]:
from sklearn.dummy import DummyRegressor

ols_mse = -cross_val_score(make_pipeline(preprocess, LinearRegression()), X, y, cv=folds,
                           scoring='neg_mean_squared_error').mean()
mean_mse = -cross_val_score(DummyRegressor(), X, y, cv=folds,
                            scoring='neg_mean_squared_error').mean()

best_ridge = np.argmin(ridge_mse)
pd.DataFrame({'10-fold CV MSE': [ols_mse, ridge_mse[best_ridge], mean_mse]},
             index=['least squares (no penalty)',
                    f'ridge, best lambda = {ridge_lambdas[best_ridge]:.2g}',
                    'predict the average']).round(0)

In [ ]:
from ch_6_and_10_demos_helpers import plot_cv_lambda

plot_cv_lambda({'Ridge': (ridge_lambdas, ridge_mse)}, ols_mse, mean_mse);

Ridge's curve dips a little below least squares, then rises steeply toward "predict the average" as $\lambda$ grows. On all 263 players, the best penalty helps only a little, about 3%. Nineteen features and 263 players isn't much flexibility to tame. **Penalties help most when there are many features for the amount of data**, and that's exactly the situation of the neural networks coming up, which can have far more weights than training rows.

In bias–variance terms: with a large $\lambda$, ridge has higher bias and lower variance; with a small $\lambda$, the reverse. Like $K$ and `max_depth`, $\lambda$ is a flexibility setting.

**Ask:** Near the bottom, the ridge curve is flat across a range of $\lambda$. Does it matter which one you pick?

<details>
<summary>Answer</summary>

Not much. As with $K$ in Chapter 5, values near the flat bottom are about equally good, and which one wins can depend on the shuffle into folds. What cross-validation reliably does is steer you away from the bad end (here, very large $\lambda$).

</details>

**Code worth reading.** As always, cross-validation is for choosing. Refit the chosen model on all the players.

In [ ]:
final_ridge = make_pipeline(preprocess, Ridge(alpha=ridge_lambdas[best_ridge]))
final_ridge.fit(X, y)
print(f'Final ridge: lambda = {ridge_lambdas[best_ridge]:.2g}, '
      f'sum of squared coefficients = {np.sum(final_ridge[-1].coef_**2):,.0f}')

**Code worth reading.** The lasso's cross-validation curve beside ridge's, with the same recipe. `max_iter=100_000` gives the lasso's step-by-step search enough steps to finish.

In [ ]:
lasso_lambdas = np.logspace(-2, 3, 21)

lasso_mse = []
for lam in lasso_lambdas:
    model = make_pipeline(preprocess, Lasso(alpha=lam, max_iter=100_000))   # more steps: fully solved
    scores = cross_val_score(model, X, y, cv=folds, scoring='neg_mean_squared_error')
    lasso_mse.append(-scores.mean())

best_lasso = np.argmin(lasso_mse)
pd.DataFrame({'10-fold CV MSE': [ols_mse, ridge_mse[best_ridge], lasso_mse[best_lasso], mean_mse]},
             index=['least squares (no penalty)',
                    f'ridge, best lambda = {ridge_lambdas[best_ridge]:.2g}',
                    f'lasso, best lambda = {lasso_lambdas[best_lasso]:.2g}',
                    'predict the average']).round(0)

In [ ]:
plot_cv_lambda({'Ridge': (ridge_lambdas, ridge_mse), 'Lasso': (lasso_lambdas, lasso_mse)},
               ols_mse, mean_mse);

The lasso's curve has the same shape, and its best value lands slightly below ridge's here (both about 3 to 4% better than least squares). Neither wins by enough to matter on 263 players; cross-validation is how you'd decide on your own data. Notice the lasso gets there with fewer features: at its best $\lambda$ it has dropped 6 of the 19.

<a id="section-10"></a>

## 10 · The same shape as pruning

In *Trees and Regularization*, **cost-complexity pruning** kept the subtree with the lowest

$$
\text{total cost} = \text{training error} + \alpha \lvert T \rvert,
$$

where $\lvert T \rvert$ is the number of leaves. Compare it with ridge. Both are **training error + strength × complexity**, and in both, a bigger strength gives a simpler model. Only the measure of complexity differs: the number of leaves for a tree, the size of the coefficients for a line. The same shape works for any model that has a size to charge for. That's what the word **regularization** covers when the fix is a penalty.

| Method | Training error | + | Strength × complexity | Strength: ISL / scikit-learn |
| --- | --- | --- | --- | --- |
| Pruning (eq. 8.4) | RSS of the tree | + | $\alpha \lvert T \rvert$: number of leaves | $\alpha$ / `ccp_alpha` |
| Ridge (eq. 6.5) | RSS of the line | + | $\lambda \sum_{j=1}^{p} \beta_j^2$: sum of squared coefficients | $\lambda$ / `alpha` |
| Lasso (eq. 6.7) | RSS of the line | + | $\lambda \sum_{j=1}^{p} \lvert \beta_j \rvert$: sum of absolute coefficients | $\lambda$ / `alpha` |

> **A note on symbols: watch the letters.** ISL and scikit-learn use different letters for these strengths, and scikit-learn reuses ISL's pruning letter for ridge and lasso.
>
> | What it controls | ISL writes | scikit-learn writes | Bigger value means |
> | --- | --- | --- | --- |
> | Pruning penalty | $\alpha$ (eq. 8.4) | `ccp_alpha` | **simpler** tree |
> | Ridge / lasso penalty | $\lambda$ (eqs. 6.5, 6.7) | `alpha` in `Ridge`, `Lasso` | **simpler** model (more shrinkage) |
> | Logistic regression penalty | — (ISL §6.2 is about regression) | `C` in `LogisticRegression` | **weaker** penalty |
>
> The last row is the one from Chapter 4: `LogisticRegression` adds a ridge-style penalty by default, with `C=1`. `C` is an *inverse* strength, so a smaller `C` means a *stronger* penalty, and `C=np.inf` turns it off, which is why the Chapter 4 notebooks used it. We'll return to it with neural networks.

<a id="section-11"></a>

## 11 · Summary, and next time

- A flexible model, such as a high-degree polynomial, can fit the training data almost perfectly and still overfit: its training error is far below its test error, and its fit changes wildly from sample to sample, most of all at the ends.
- Ridge keeps the flexibility but charges for big coefficients: **RSS + $\lambda \sum_{j=1}^{p} \beta_j^2$.** It shrinks every coefficient toward zero, which calms an overfit fit: it trades a little bias for a lot less variance. A second benefit: it steadies coefficients that the data can't pin down, like those of two near-copy features.
- With two coefficients, the loss RSS + $\lambda \times$ penalty is one bowl, and ridge's answer is its bottom. As $\lambda$ grows, the bottom moves along a curved path from least squares toward $(0, 0)$.
- The lasso charges the absolute values instead, and sets some coefficients exactly to zero, so it also selects features. In the budget picture, ridge keeps the coefficients inside a circle and the lasso inside a diamond. The answer is where the smallest RSS oval reaches the region, and the diamond's corners sit on the axes.
- Standardize the features before ridge or lasso, so the penalty treats them fairly. Trees don't need it.
- Choose the strength by cross-validation, then refit on all the data. Watch the letters: ISL's $\lambda$ is scikit-learn's `alpha`, and `C` runs the other way.
- Pruning, ridge, and lasso share one shape: **training error + strength × complexity.**
- **Next:** neural networks, starting with a single neuron. A network is a very flexible model, so it can overfit too. When we see that happen, we'll come back to this penalty: ridge's penalty on a network's weights is called **weight decay**.

# Practice Questions

These questions cover polynomial overfitting, ridge and lasso, standardizing, choosing $\lambda$, and the notation. Questions like these will appear on the learning check, so try each one on paper before you open its hint or answer.

For reference:

- RSS $= \sum_{i=1}^{n} (y_i - \hat y_i)^2$. 
- Ridge: Loss = RSS $+ \lambda \sum_{j=1}^{p} \beta_j^2$. 
- Lasso: Loss = RSS $+ \lambda \sum_{j=1}^{p} \lvert \beta_j \rvert$. 
- Pruning: Loss = training error $+ \alpha \lvert T \rvert$.

### Question 1: Degrees

You fit polynomials of degree 1, 3, and 12 to 20 points. (a) Which has the lowest training error? (b) Which is likely to have the highest test error, and why? (c) Is a degree-12 polynomial a linear regression? (See [Section 1](#section-1) and [Section 2](#section-2).)

<details>
<summary>Answer</summary>

(a) **Degree 12.** More flexibility can only lower the training error.

(b) **Degree 12** (assuming the true curve is smooth). It fits the noise in the 20 points, and between and beyond them it swings wildly.

(c) **Yes.** It's a linear combination of features ($x, x^2, \dots, x^{12}$), and the coefficients are fit by least squares.

</details>

### Question 2: Two symptoms

Name two ways to see that a model is overfitting. (See [Section 2](#section-2).)

<details>
<summary>Answer</summary>

**Training error far below test error**, and **fits (model coefficients) that change a lot from one sample of training data to another** (high variance).

</details>

### Question 3: Read the formula

In ridge, Loss = RSS $+ \lambda \sum_{j=1}^{p} \beta_j^2$: (a) which term rewards fitting the training points well? (b) Which term charges for big coefficients? (c) What model do you get at $\lambda = 0$, and what do you get at a huge $\lambda$? (See [Section 3](#section-3).)

<details>
<summary>Hint for the second part of (c)</summary>

Look closely at what $j$ runs over in the penalty: $j = 1$ to $p$. Which coefficients does the penalty charge for, and which one does it leave out? If a huge $\lambda$ pushes every coefficient in that sum to nearly zero, what's left of the prediction $\hat y = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$?

</details>

<details>
<summary>Answer</summary>

(a) **The RSS.** (b) **The penalty $\lambda \sum_{j=1}^{p} \beta_j^2$.**

(c) **Ordinary least squares** at $\lambda = 0$ (no penalty). At a huge $\lambda$, **about the average response for everyone:** every coefficient is shrunk to nearly zero, leaving only the intercept, which isn't penalized.

</details>

### Question 4: Standardize first

(a) A feature is measured in dollars. You change it to thousands of dollars. What happens to its least squares coefficient?

(b) Why does that matter for ridge, and what do we do about it?

(c) Does a decision tree need standardized features? (See [Section 5](#section-5).)

<details>
<summary>Hint</summary>

A coefficient is the change in the prediction per one unit of the feature. How big is one unit now?

</details>

<details>
<summary>Answer</summary>

(a) **It gets 1,000 times bigger**, because one unit of the feature is now 1,000 times bigger.

(b) The penalty charges every coefficient the same, so a feature's units would decide how hard it gets penalized. **Standardize** every feature first, so each is measured in standard deviations.

(c) **No.** A split only asks which side of $s$ a point is on, and changing units moves $s$ along with the points.

</details>

### Question 5: Near-copies

Two features are nearly identical. (a) What does least squares do with their coefficients from one sample to the next? (b) What does ridge do? (c) Which kind of error does ridge accept to get that? (See [Section 6](#section-6).)

<details>
<summary>Hint</summary>

What can the data tell you: how much the two features matter together, or how to split the credit between them?

</details>

<details>
<summary>Answer</summary>

(a) **They swing wildly** (one large and positive, the other large and negative, for example), though their sum stays about right.

(b) **It splits the credit about evenly and stays steady.**

(c) **A little bias:** the coefficients are pulled slightly toward zero. In return, the variance drops a lot.

</details>

### Question 6: The budget picture

In the budget picture of ridge and the lasso: (a) A classmate says, "Ridge picks the point in the circle closest to least squares." What's wrong with that? (b) When least squares is outside the region, why is the best model always on the region's edge, never inside it? (c) Why does the lasso's best model often land on an axis, while ridge's almost never does? (d) Does a bigger budget $s$ act like a bigger or a smaller $\lambda$? (See [Demo M](#demo-m).)

<details>
<summary>Hint</summary>

Think about the hollow point in Demo M. For (b), from a model inside the region, which way could you step to lower the RSS, and would you still be within the budget?

</details>

<details>
<summary>Answer</summary>

(a) **Ridge picks the lowest-RSS point, not the closest one.** It's where the smallest oval first reaches the circle, and that depends on the ovals' shape, so it's usually a different point.

(b) **From a model inside the region, a small step toward least squares lowers the RSS** and still fits the budget. So an inside point can't be the best; the best model spends the whole budget.

(c) **The diamond's corners sit on the axes** and stick out, so an oval often reaches a corner first, and at a corner one coefficient is exactly 0. A circle has no corners, so the first touch is almost never exactly on an axis.

(d) **A smaller $\lambda$.** A bigger budget allows bigger coefficients, so there's less shrinking. A budget big enough to include least squares is the same as $\lambda = 0$.

</details>

### Question 7: Pick the smallest loss?

You fit ridge with one coefficient at two values of $\lambda$. At ridge's answer, the loss is 8.1 when $\lambda = 0$ and 89.7 when $\lambda = 27$. A classmate concludes, "$\lambda = 0$ is better, because its loss is smaller." (a) What data is the loss computed from? (b) Give two reasons the classmate's argument doesn't work. (c) How should $\lambda$ be chosen instead? (See [Demo K](#demo-k) and [Section 9](#section-9).)

<details>
<summary>Hint</summary>

Would a smaller training loss always mean better predictions on new data? And is the loss at $\lambda = 27$ even the same formula as the loss at $\lambda = 0$?

</details>

<details>
<summary>Answer</summary>

(a) **The training data.**

(b) **First, a smaller training loss doesn't mean better predictions on new data;** the least squares fit ($\lambda = 0$) has the smallest training error, and that's exactly how overfitting happens. **Second, the two losses are different formulas:** the $\lambda = 27$ loss includes a penalty term that the $\lambda = 0$ loss doesn't, so its lowest value is bound to be larger. Comparing them is comparing scores from two different tests.

(c) **By cross-validation:** for each $\lambda$, fit on part of the data and measure the error (with no penalty) on the held-out part. Pick the $\lambda$ with the lowest cross-validation error.

</details>

### Question 8: Ridge or lasso?

(a) Which method can set coefficients exactly to zero?

(b) After fitting at some $\lambda$, a model has 7 of its 19 coefficients exactly equal to 0. Was it ridge or the lasso?

(c) What is this \"free\" feature selection called, as a family of feature-selection methods? (See [Demo G](#demo-g).)

<details>
<summary>Answer</summary>

(a) **The lasso.** (b) **The lasso:** ridge shrinks coefficients toward zero but doesn't set them exactly to zero. (c) **An embedded method:** the selection happens as part of fitting.

</details>

### Question 9: Translate the letters

For each, give the scikit-learn name. (See the note on symbols in [Section 10](#section-10).)

(a) ISL's pruning strength $\alpha$.

(b) ISL's ridge strength $\lambda$.

(c) True or false: in `LogisticRegression`, making `C` smaller makes the penalty weaker.

<details>
<summary>Answer</summary>

(a) **`ccp_alpha`.** (b) **`alpha`** (in `Ridge`).

(c) **False.** `C` is an inverse strength: a smaller `C` means a *stronger* penalty.

</details>

### Question 10: Choose $\lambda$

10-fold cross-validation for ridge gives:

| $\lambda$ | 0.1 | 1 | 3 | 10 | 100 |
| --- | --- | --- | --- | --- | --- |
| CV MSE | 116,000 | 113,500 | 111,700 | 115,400 | 136,400 |

(a) Which $\lambda$ would you choose?

(b) What do you do next, before predicting new players' salaries? (See [Demo I](#demo-i).)

<details>
<summary>Answer</summary>

(a) **$\lambda = 3$**, the lowest CV MSE.

(b) **Refit ridge with $\lambda = 3$ on all the training data**, and use that model. Cross-validation was only for choosing.

</details>

### Question 11: $\lambda$ and bias–variance

On data like Hitters, which has higher bias, ridge with $\lambda = 1{,}000$ or ridge with $\lambda = 0.01$? Which has higher variance? (See [Demo I](#demo-i).)

<details>
<summary>Answer</summary>

**$\lambda = 1{,}000$ has higher bias; $\lambda = 0.01$ has higher variance.** A large $\lambda$ shrinks the coefficients hard, so the method is rigid; a tiny $\lambda$ is nearly least squares, the flexible end.

</details>

### Question 12: Same shape

Match each complexity charge to its method: pruning, ridge, or lasso. (See [Section 10](#section-10).)

(a) $\lambda \sum_{j=1}^{p} \lvert \beta_j \rvert$ &emsp; (b) $\alpha \lvert T \rvert$ &emsp; (c) $\lambda \sum_{j=1}^{p} \beta_j^2$

<details>
<summary>Answer</summary>

(a) **Lasso** (sum of absolute coefficients). (b) **Pruning** (number of leaves). (c) **Ridge** (sum of squared coefficients).

</details>